# `latincy_vocab` — a standalone spaCy/LatinCy vocabulary component

`latincy_vocab` consumes an already-parsed spaCy `Doc` and sets `doc._.vocab_list`
(a `VocabList`) — a deduplicated, reading-order vocabulary list. It **consumes upstream
token annotations** (lemma, POS, and `token._.gloss` from latincy-lexicon's
`whitakers_words`) rather than loading any gloss/lexicon files itself.

- Proper names are excluded (they route to a separate NER/NEL channel).
- Standalone enclitics (`-que`) are dropped.
- Glosses are optional: with no upstream gloss pipe, the list is still produced (lexicon-free).

In [1]:
import spacy
import vocabbuilder  # importing registers the `latincy_vocab` factory
from vocabbuilder import VocabList

PASSAGE = (
    "Q. Mucius augur multa narrare de C. Laelio socero suo memoriter et iucunde "
    "solebat nec dubitare illum in omni sermone appellare sapientem; ego autem a "
    "patre ita eram deductus ad Scaeuolam sumpta uirili toga, ut, quoad possem et "
    "liceret, a senis latere numquam discederem."
)

## 1. Lexicon-free — `nlp.add_pipe("latincy_vocab")`

Just add the component to any LatinCy pipeline. No gloss data needed.

In [2]:
nlp = spacy.load("la_core_web_lg")
nlp.add_pipe("latincy_vocab")

doc = nlp(PASSAGE)
vocab = doc._.vocab_list
assert isinstance(vocab, VocabList)

print(f"{len(vocab)} entries (deduped, no PROPN, no enclitics)\n")
for e in vocab.by_first_occurrence():
    print(f"{e.display_lemma:<14} {e.pos:<6} x{e.frequency}")

37 entries (deduped, no PROPN, no enclitics)

augur          NOUN   x1
multus         DET    x1
narro          VERB   x1
de             ADP    x1
socer          NOUN   x1
suus           DET    x1
memoriter      ADV    x1
et             CCONJ  x2
iucunde        ADV    x1
soleo          VERB   x1
neque          CCONJ  x1
dubito         VERB   x1
ille           DET    x1
in             ADP    x1
omnis          DET    x1
sermo          NOUN   x1
appello        VERB   x1
sapiens        ADJ    x1
ego            PRON   x1
autem          CCONJ  x1
ab             ADP    x2
pater          NOUN   x1
ita            ADV    x1
sum            AUX    x1
deduco         VERB   x1
ad             ADP    x1
sumo           VERB   x1
virilis        ADJ    x1
toga           NOUN   x1
ut             SCONJ  x1
quoad          SCONJ  x1
possum         VERB   x1
licet          VERB   x1
senex          NOUN   x1
latus          NOUN   x1
numquam        ADV    x1
discedo        VERB   x1


## 2. Glosses & citation forms — consume `whitakers_words` upstream

Add latincy-lexicon's `whitakers_words` **before** `latincy_vocab`; the component picks up
each token's `token._.gloss` **and** `token._.lexicon`, producing textbook citation forms via
`latincy_lexicon.format_principal_parts` — `narro, narrare, narravi, narratum, v., tell…`.
Nouns show gender (`toga, togae, f.`); verbs/adjectives/adverbs get a POS tag.

In [3]:
from pathlib import Path
import latincy_lexicon

# Resolve the lexicon data shipped with the installed (editable) latincy-lexicon.
LEX = Path(latincy_lexicon.__file__).resolve().parents[2] / "data" / "json"

nlp_g = spacy.load("la_core_web_lg")
nlp_g.add_pipe(
    "whitakers_words",
    config={"lexicon_path": str(LEX / "lexicon.json"), "analyzer_path": str(LEX / "analyzer.json")},
)
nlp_g.add_pipe("latincy_vocab")
print("pipeline:", nlp_g.pipe_names, "\n")

# entry.formatted() → "headword, marker, gloss" (gender for nouns; v./adj./adv. for the rest)
glossed = nlp_g(PASSAGE)._.vocab_list
for e in glossed.by_first_occurrence():
    print(e.formatted())

pipeline: ['enclitic_splitter', 'tok2vec', 'senter', 'token_fix', 'normer', 'tagger', 'morphologizer', 'trainable_lemmatizer', 'lookup_lemmatizer', 'uv_normalizer', 'harmonizer', 'remorpher', 'parser', 'ner', 'whitakers_words', 'latincy_vocab'] 



augur, auguris, augur, one who interprets behavior of birds
multus, -a, -um, adj., much, many, great, many a
narro, narrare, narravi, narratum, v., tell, tell about, relate, narrate, recount, describe
de, prep., down/away from, from, off
socer, soceri, m., father in law
suus, sui, m., his/one's (own), her (own), hers, its (own)
memoriter, adv., from memory
et, conj., and, and even
iucunde, adv.
soleo, solere, soliti, v., be in the habit of
neque, conj., nor, and..not
dubito, dubitare, dubitavi, dubitatum, v., doubt
ille, adj., that
in, prep., in, on, at (space)
omnis, -e, adj., each, every, every one (of a number)
sermo, sermonis, m., conversation, discussion
appello, appellare, appellavi, appellatum, v., call (upon)
sapiens, sapientis, adj., rational
ego, pron., I, me (PERS)
autem, conj., but (postpositive), on the other hand/contrary
ab, prep., by (agent), from (departure, cause, remote origin/time)
pater, patri, m., father
ita, adv., thus, so
sum, v., be
deduco, deducere, deduxi, de

## 3. Views & export

`VocabList` offers `by_frequency()`, `by_alpha()`, `by_first_occurrence()`,
`filter_pos()`, `filter_min_frequency()`, plus `to_markdown()` / `to_json()` / `to_dicts()`.

In [4]:
# Nouns and verbs only, as a Markdown glossary
content = glossed.filter_pos({"NOUN", "VERB"}).by_alpha()
print(content.to_markdown())

- **appello, appellare, appellavi, appellatum**, v., call (upon)
- **augur, auguris**, augur, one who interprets behavior of birds
- **deduco, deducere, deduxi, deductum**, v., lead/draw//pull/bring/stretch down/away/out/off
- **discedo, discedere, discessi, discessum**, v., go/march off, depart, withdraw
- **dubito, dubitare, dubitavi, dubitatum**, v., doubt
- **latus, lati, n.**, side
- **liceo, licere, licui, licitum**, v., it is permitted, one may
- **narro, narrare, narravi, narratum**, v., tell, tell about, relate, narrate, recount, describe
- **pater, patri, m.**, father
- **possum**, v., be able, can
- **senex, senis, m.**, old man
- **sermo, sermonis, m.**, conversation, discussion
- **socer, soceri, m.**, father in law
- **soleo, solere, soliti**, v., be in the habit of
- **sumo, sumere, sumpsi, sumptum**, v., take up
- **toga, togae, f.**, toga


In [5]:
# JSON for downstream consumers (e.g. the reader's substrate)
print(glossed.by_frequency().to_json()[:600], "...")

[
  {
    "lemma": "et",
    "display_lemma": "et",
    "citation_form": null,
    "pos": "CCONJ",
    "glosses": [
      "and, and even"
    ],
    "forms_seen": [
      "et"
    ],
    "frequency": 2,
    "morphology": [],
    "passage_indices": [
      0,
      1
    ],
    "first_index": 7
  },
  {
    "lemma": "ab",
    "display_lemma": "ab",
    "citation_form": null,
    "pos": "ADP",
    "glosses": [
      "by (agent), from (departure, cause, remote origin/time)"
    ],
    "forms_seen": [
      "a"
    ],
    "frequency": 2,
    "morphology": [],
    "passage_indices": [
      1,
     ...
